In [ ]:
///- --package . --timeout 1200000
:< sm'_operators
:< xsd
:< xsd_emit

:! repo_path = string

:= repo_text ((repo_path text) : repo_path) : string = text

:= repo_file (path : repo_path) : string = fiscal_paths::under (repo_text path)

:| schema_source =
    | Xsd : list (string * repo_path)
    | JsonSchema

:| document_source =
    | Listed : list string
    | Manifest : repo_path

:= catalog_of (entries : list (string * repo_path)) : xsd_resolve::schema_catalog =
    listm::map (\ (location, path) => location, repo_file path) entries

:| family_extra =
    | Signed
    | CfdiComplements
    | ModelChecks : string
    | DocumentDefect : string * string
    | GatePreload : repo_path
    | GateResolve
    | Ungated : string
    | LifecycleEvent : string

:~ ^ has_extra (extra : family_extra) (extras : list family_extra) : bool =
    ?= extras ?|
    | x :: rest => x = extra || has_extra extra rest
    | [] => !-

:~ ^ model_check_texts (extras : list family_extra) : string =
    ?= extras ?|
    | (ModelChecks text) :: rest => text ++# model_check_texts rest
    | _ :: rest => model_check_texts rest
    | [] => ""

:~ ^ document_defects (extras : list family_extra) : list (string * string) =
    ?= extras ?|
    | (DocumentDefect defect) :: rest => defect :: document_defects rest
    | _ :: rest => document_defects rest
    | [] => []

:~ ^ gate_preloads (extras : list family_extra) : list string =
    ?= extras ?|
    | (GatePreload path) :: rest => repo_text path :: gate_preloads rest
    | _ :: rest => gate_preloads rest
    | [] => []

:~ ^ lifecycle_events (extras : list family_extra) : list string =
    ?= extras ?|
    | (LifecycleEvent code) :: rest => code :: lifecycle_events rest
    | _ :: rest => lifecycle_events rest
    | [] => []

:~ ^ ungated_reason (extras : list family_extra) : option string =
    ?= extras ?|
    | (Ungated reason) :: _ => Some reason
    | _ :: rest => ungated_reason rest
    | [] => None

:= xmldsig_schema () : repo_path = repo_path "deps/corpus/eu/ubl21/common/UBL-xmldsig-core-schema-2.1.xsd"

:= resolved_gate () : list family_extra = [ GateResolve; GatePreload (xmldsig_schema ()) ]

:! module_id = string

:= module_text ((module_id text) : module_id) : string = text

:== family_root =
    {
        module_name : module_id
        schema_path : repo_path
        root_element : tag_name
        root_type : definition_name written
        target_namespace : namespace_uri
        source : schema_source
        known_defect : option string
        documents : document_source
        ubl_json_documents : document_source
        extras : list family_extra
    }

:= root_type_name (family : family_root) : string = type_name (definition_text family::root_type)

:= bare_family
    (module : module_id) (schema : repo_path) (root : tag_name) (ty : definition_name written) (space : namespace_uri)
    : family_root
    =
    {
        module_name = module
        schema_path = schema
        root_element = root
        root_type = ty
        target_namespace = space
        source = Xsd []
        known_defect = None
        documents = Listed []
        ubl_json_documents = Listed []
        extras = []
    }

:= cfdi_v4_0 () : family_root =
    {
        module_name = module_id "generated_cfdi_v4_0"
        schema_path = repo_path "deps/corpus/mx/cfdi/cfdv40.xsd"
        root_element = tag_name "Comprobante"
        root_type = definition_name "Comprobante"
        target_namespace = namespace_uri "http://www.sat.gob.mx/cfd/4"
        source =
            Xsd [
                (
                    "http://www.sat.gob.mx/sitio_internet/cfd/catalogos/catCFDI.xsd",
                    repo_path "deps/corpus/mx/cfdi/catCFDI.xsd"
                )
                (
                    "http://www.sat.gob.mx/sitio_internet/cfd/tipoDatos/tdCFDI/tdCFDI.xsd",
                    repo_path "deps/corpus/mx/cfdi/tdCFDI.xsd"
                )
            ]
        known_defect = None
        documents = Listed [ "deps/corpus/mx/stamped/cfdiutils-cfdi40-valid.xml" ]
        ubl_json_documents = Listed []
        extras =
            [
                CfdiComplements
                GatePreload (repo_path "deps/corpus/mx/cfdi/catCFDI.xsd")
                GatePreload (repo_path "deps/corpus/mx/cfdi/tdCFDI.xsd")
            ]
    }

:= cfdi_complement
    (module_name : string) (file : string) (root : string) (space : string) (catalog : list (string * repo_path))
    : family_root
    =
    {
        module_name = module_id module_name
        schema_path = repo_path ("deps/corpus/mx/complements/" ++# file)
        root_element = tag_name root
        root_type = definition_name root
        target_namespace = namespace_uri space
        source = Xsd catalog
        known_defect = None
        documents = Listed []
        ubl_json_documents = Listed []
        extras =
            [
                GateResolve
                GatePreload (repo_path "deps/corpus/eu/ubl21/common/UBL-xmldsig-core-schema-2.1.xsd")
            ]
    }

:= td_cfdi_import () : string * repo_path =
    "http://www.sat.gob.mx/sitio_internet/cfd/tipoDatos/tdCFDI/tdCFDI.xsd", repo_path "deps/corpus/mx/cfdi/tdCFDI.xsd"

:= cat_cfdi_import () : string * repo_path =
    "http://www.sat.gob.mx/sitio_internet/cfd/catalogos/catCFDI.xsd", repo_path "deps/corpus/mx/cfdi/catCFDI.xsd"

:= cat_pagos_import () : string * repo_path =
    "http://www.sat.gob.mx/sitio_internet/cfd/catalogos/Pagos/catPagos.xsd",
    repo_path "deps/corpus/mx/complements/catPagos.xsd"

:= cfdi_tfd_1_1 () : family_root =
    cfdi_complement
        "generated_cfdi_tfd_1_1"
        "TimbreFiscalDigitalv11.xsd"
        "TimbreFiscalDigital"
        "http://www.sat.gob.mx/TimbreFiscalDigital"
        [ td_cfdi_import () ]

:= cfdi_pagos_2_0 () : family_root =
    cfdi_complement
        "generated_cfdi_pagos_2_0"
        "Pagos20.xsd"
        "Pagos"
        "http://www.sat.gob.mx/Pagos20"
        [ cat_cfdi_import (); td_cfdi_import (); cat_pagos_import () ]

:= complements_of (family : family_root) : list (string * family_root) =
    ? has_extra CfdiComplements family::extras
    ?> [ ("TimbreFiscalDigital11", cfdi_tfd_1_1 ()); ("Pagos20", cfdi_pagos_2_0 ()) ]
    |: []

:= signed_family (module_name : string) (schema_path : string) (root : string) (root_type : string) (space : string)
    : family_root
    =
    {
        module_name = module_id module_name
        schema_path = repo_path schema_path
        root_element = tag_name root
        root_type = definition_name root_type
        target_namespace = namespace_uri space
        source = Xsd []
        known_defect = None
        documents = Listed []
        ubl_json_documents = Listed []
        extras = [ Signed ]
    }

:= dfe_rows () : list (string * string * string * string * string) =
    [
        ("generated_nfe_v2_00", "NFe.AppTeste/Schemas/nfe_v2.00.xsd", "NFe", "TNFe", "nfe");
        ("generated_nfe_v3_10", "NFe.AppTeste/Schemas/nfe_v3.10.xsd", "NFe", "TNFe", "nfe");
        ("generated_cte_v2_00", "CTe.AppTeste/schemas/cte_v2.00.xsd", "CTe", "TCTe", "cte");
        ("generated_cte_v3_00", "CTe.AppTeste/schemas/cte_v3.00.xsd", "CTe", "TCTe", "cte");
        ("generated_cte_v4_00", "CTe.AppTeste/schemas/cte_v4.00.xsd", "CTe", "TCTe", "cte");
        ("generated_cte_os_v3_00", "CTe.AppTeste/schemas/cteOS_v3.00.xsd", "CTeOS", "TCTeOS", "cte");
        ("generated_cte_os_v4_00", "CTe.AppTeste/schemas/cteOS_v4.00.xsd", "CTeOS", "TCTeOS", "cte");
        ("generated_cte_simp_v4_00", "CTe.AppTeste/schemas/cteSimp_v4.00.xsd", "CTeSimp", "TCTeSimp", "cte");
        ("generated_gtve_v3_00", "CTe.AppTeste/schemas/GTVe_v3.00.xsd", "GTVe", "TGTVe", "cte");
        ("generated_gtve_v4_00", "CTe.AppTeste/schemas/GTVe_v4.00.xsd", "GTVe", "TGTVe", "cte");
        ("generated_mdfe_v1_00", "MDFe.AppTeste/Schemas/mdfe_v1.00.xsd", "MDFe", "TMDFe", "mdfe");
        ("generated_mdfe_v3_00", "MDFe.AppTeste/Schemas/mdfe_v3.00.xsd", "MDFe", "TMDFe", "mdfe")
    ]

:= dfe_root (row : string * string * string * string * string) : family_root =
    := module_name, schema, root, root_type, portal = row
    := space = "http://www.portalfiscal.inf.br/" ++# portal
    signed_family module_name ("deps/corpus/DFe.NET/" ++# schema) root root_type space

:= family_roots () : list family_root = listm::map dfe_root (dfe_rows ())

:= nfe_model_checks () : string =
    "\n///> rust\ntest generated_nfe_model_type_checks =\n"
    ++# "    match x_t_cod_uf_ibge_of_string \"35\" with\n"
    ++# (
        "    | Some code => if x_t_cod_uf_ibge_to_string code <> \"35\" then failwith \"TCodUfIBGE 35 round "
        ^^ "trip\"\n"
    )
    ++# "    | None => failwith \"35 parses as TCodUfIBGE\"\n"
    ++# "    if x_t_cnpj_is_valid \"12345678000195\" = false then failwith \"a 14-digit CNPJ is valid\"\n"
    ++# "    if x_t_cnpj_is_valid \"1234567800019\" then failwith \"a 13-digit CNPJ is not\"\n"
    ++# (
        "    match xml.read_xml \"<enderEmit><xLgr>Rua A</xLgr><nro>10</nro><xBairro>Centro</xBairro><cMun>3550"
        ^^ "308</cMun><xMun>Sao Paulo</xMun><UF>SP</UF><CEP>01001000</CEP><cPais>1058</cPais></enderEmit>\" with\n"
    )
    ++# "    | node :: [] =>\n"
    ++# "        inl address = xml_codec.expect_decoded (x_t_ender_emi_of_xml node)\n"
    ++# (
        "        if x_t_ender_emi_x_mun_to_string address.x_mun <> \"Sao Paulo\" || x_t_ender_emi_cep_to_string "
        ^^ "address.cep <> \"01001000\" then failwith \"TEnderEmi text fields\"\n"
    )
    ++# "        if x_t_uf_emi_to_string address.uf <> \"SP\" then failwith \"TEnderEmi UF enumeration\"\n"
    ++# (
        "        if xml.same_node node (x_t_ender_emi_to_xml \"enderEmit\" address) = false then failwith "
        ^^ "\"TEnderEmi XML round trip\"\n"
    )
    ++# "        match address.c_pais with\n"
    ++# "        | Some _ => ()\n"
    ++# "        | None => failwith \"TEnderEmi optional cPais\"\n"
    ++# "    | _ => failwith \"one enderEmit element\"\n"

:= nfe_v4_base () : family_root =
    bare_family
        (module_id "generated_nfe_v4")
        (repo_path "deps/corpus/DFe.NET/NFe.AppTeste/Schemas/nfe_v4.00.xsd")
        (tag_name "NFe")
        (definition_name "TNFe")
        (namespace_uri "http://www.portalfiscal.inf.br/nfe")

:= nfe_v4_00 () : family_root =
    := base = nfe_v4_base ()
    { base with extras = [ Signed; ModelChecks (nfe_model_checks ()) ] }

:= nfe_event_rows () : list (string * string * string) =
    [
        ("generated_nfe_evento_canc_v1_00", "procEventoCancNFe_v1.00.xsd", "110111");
        ("generated_nfe_evento_cce_v1_00", "procCCeNFe_v1.00.xsd", "110110");
        ("generated_nfe_evento_conf_recebto_v1_00", "procConfRecebtoNFe_v1.00.xsd", "210200")
    ]

:= nfe_event_base (module_name : string) (schema : string) : family_root =
    := path = "deps/corpus/DFe.NET/NFe.AppTeste/Schemas/" ++# schema
    signed_family module_name path "procEventoNFe" "TProcEvento" "http://www.portalfiscal.inf.br/nfe"

:= nfe_event (row : string * string * string) : family_root =
    := module_name, schema, code = row
    := base = nfe_event_base module_name schema
    { base with extras = [ Signed; LifecycleEvent code ] }

:= nfe_events () : list family_root = listm::map nfe_event (nfe_event_rows ())

:~ ^ manifest_entries (prefix : string) (entries : list string) (acc : list string) : list string =
    ?= entries ?|
    | entry :: rest =>
        := trimmed = sm'::trim entry
        manifest_entries prefix rest (? trimmed = "" ?> acc |: (prefix ++# trimmed) :: acc)
    | [] => listm::rev acc

:= manifest_documents (prefix : string) (manifest : string) : list string =
    := path = fiscal_paths::under manifest
    ? xml::is_file path = !-
    ?> []
    |: manifest_entries prefix (sm'::split_list "\n" (xml::read_text_replacing_invalid_utf8 path)) []

:= documents_of (source : document_source) : list string =
    ?= source ?|
    | Listed paths => paths
    | Manifest manifest => manifest_documents "deps/corpus/" (repo_text manifest)

:= ubl_invoice_2_1 () : family_root =
    {
        module_name = module_id "generated_ubl_invoice_2_1"
        schema_path = repo_path "deps/corpus/eu/ubl21/maindoc/UBL-Invoice-2.1.xsd"
        root_element = tag_name "Invoice"
        root_type = definition_name "InvoiceType"
        target_namespace = namespace_uri "urn:oasis:names:specification:ubl:schema:xsd:Invoice-2"
        source = Xsd []
        known_defect = None
        documents = Manifest (repo_path "deps/corpus/ubl-invoice-documents.txt")
        ubl_json_documents = Manifest (repo_path "deps/corpus/ubl-invoice-json-documents.txt")
        extras =
            [
                DocumentDefect ("deps/corpus/my/samples/1.1-Invoice-ForeignCurrency-Sample.xml", "PayableAmount: abc")
                GateResolve
                GatePreload (repo_path "deps/corpus/eu/ubl21/common/UBL-xmldsig-core-schema-2.1.xsd")
            ]
    }

:= ubl_creditnote_2_1 () : family_root =
    {
        module_name = module_id "generated_ubl_creditnote_2_1"
        schema_path = repo_path "deps/corpus/eu/ubl21/maindoc/UBL-CreditNote-2.1.xsd"
        root_element = tag_name "CreditNote"
        root_type = definition_name "CreditNoteType"
        target_namespace = namespace_uri "urn:oasis:names:specification:ubl:schema:xsd:CreditNote-2"
        source = Xsd []
        known_defect = None
        documents = Manifest (repo_path "deps/corpus/ubl-creditnote-documents.txt")
        ubl_json_documents = Manifest (repo_path "deps/corpus/ubl-creditnote-json-documents.txt")
        extras =
            [
                GateResolve
                GatePreload (repo_path "deps/corpus/eu/ubl21/common/UBL-xmldsig-core-schema-2.1.xsd")
            ]
    }

:= ubl_manifest (lower : string) (suffix : string) : document_source =
    Manifest (repo_path ("deps/corpus/ubl-" ++# lower ++# suffix))

:= ubl_schema (kind : string) : repo_path = repo_path ("deps/corpus/eu/ubl21/maindoc/UBL-" ++# kind ++# "-2.1.xsd")

:= ubl_namespace (kind : string) : namespace_uri =
    namespace_uri ("urn:oasis:names:specification:ubl:schema:xsd:" ++# kind ++# "-2")

:= ubl_base (kind : string) (lower : string) : family_root =
    := module = module_id ("generated_ubl_" ++# lower ++# "_2_1")
    bare_family module (ubl_schema kind) (tag_name kind) (definition_name (kind ++# "Type")) (ubl_namespace kind)

:= ubl_document_2_1 (kind : string) (lower : string) : family_root =
    := base = ubl_base kind lower
    { base with
        documents = ubl_manifest lower "-documents.txt"
        ubl_json_documents = ubl_manifest lower "-json-documents.txt"
        extras = resolved_gate ()
    }

:= ubl_business_documents_2_1 () : list family_root =
    [
        ubl_document_2_1 "DebitNote" "debitnote"; ubl_document_2_1 "ApplicationResponse" "applicationresponse";
        ubl_document_2_1 "DespatchAdvice" "despatchadvice"; ubl_document_2_1 "ReceiptAdvice" "receiptadvice"
    ]

:= fatturapa_1_2_3 () : family_root =
    {
        module_name = module_id "generated_fatturapa_1_2_3"
        schema_path = repo_path "deps/corpus/it/fatturapa/Schema_VFPR12_v1.2.3.xsd"
        root_element = tag_name "FatturaElettronica"
        root_type = definition_name "FatturaElettronicaType"
        target_namespace = namespace_uri "http://ivaservizi.agenziaentrate.gov.it/docs/xsd/fatture/v1.2"
        source =
            Xsd [
                (
                    "http://www.w3.org/TR/2002/REC-xmldsig-core-20020212/xmldsig-core-schema.xsd",
                    repo_path "deps/corpus/eu/ubl21/common/UBL-xmldsig-core-schema-2.1.xsd"
                )
            ]
        known_defect = None
        documents = Listed []
        ubl_json_documents = Listed []
        extras =
            [
                GatePreload (repo_path "deps/corpus/eu/ubl21/common/UBL-xmldsig-core-schema-2.1.xsd")
            ]
    }

:= ksef_fa_3 () : family_root =
    {
        module_name = module_id "generated_ksef_fa_3"
        schema_path = repo_path "deps/corpus/pl/ksef/schemat_FA3.xsd"
        root_element = tag_name "Faktura"
        root_type = definition_name "Faktura"
        target_namespace = namespace_uri "http://crd.gov.pl/wzor/2025/06/25/13775/"
        source =
            Xsd [
                (
                    "http://crd.gov.pl/xml/schematy/dziedzinowe/mf/2022/01/05/eD/DefinicjeTypy/",
                    repo_path "deps/corpus/pl/ksef/"
                )
            ]
        known_defect = None
        documents = Listed []
        ubl_json_documents = Listed []
        extras =
            [
                GatePreload (repo_path "deps/corpus/pl/ksef/ElementarneTypyDanych_v10-0E.xsd")
                GatePreload (repo_path "deps/corpus/pl/ksef/KodyKrajow_v10-0E.xsd")
                GatePreload (repo_path "deps/corpus/pl/ksef/StrukturyDanych_v10-0E.xsd")
            ]
    }

:= chile_dte_v10 () : family_root =
    {
        module_name = module_id "generated_cl_dte_v10"
        schema_path = repo_path "deps/corpus/cl/dte/DTE_v10.xsd"
        root_element = tag_name "DTE"
        root_type = definition_name "DTEDefType"
        target_namespace = namespace_uri "http://www.sii.cl/SiiDte"
        source = Xsd []
        known_defect = None
        documents = Listed []
        ubl_json_documents = Listed []
        extras = [ Signed; GateResolve ]
    }

:= chile_envio_dte_v10 () : family_root =
    {
        module_name = module_id "generated_cl_envio_dte_v10"
        schema_path = repo_path "deps/corpus/cl/dte/EnvioDTE_v10.xsd"
        root_element = tag_name "EnvioDTE"
        root_type = definition_name "EnvioDTE"
        target_namespace = namespace_uri "http://www.sii.cl/SiiDte"
        source = Xsd []
        known_defect = None
        documents = Listed [ "deps/corpus/cl/examples/F60T33-ejemplo.xml" ]
        ubl_json_documents = Listed []
        extras = [ Signed; GateResolve ]
    }

:= facturae_3_2_2 () : family_root =
    {
        module_name = module_id "generated_es_facturae_3_2_2"
        schema_path = repo_path "deps/corpus/es/facturae/Facturaev3_2_2.xsd"
        root_element = tag_name "Facturae"
        root_type = definition_name "Facturae"
        target_namespace = namespace_uri "http://www.facturae.gob.es/formato/Versiones/Facturaev3_2_2.xml"
        source =
            Xsd [
                (
                    "http://www.w3.org/TR/xmldsig-core/xmldsig-core-schema.xsd",
                    repo_path "deps/corpus/eu/ubl21/common/UBL-xmldsig-core-schema-2.1.xsd"
                )
            ]
        known_defect = None
        documents =
            Listed [
                "deps/corpus/es/facturae/samples/corrective.xml"
                "deps/corpus/es/facturae/samples/credit-note.xml"
                "deps/corpus/es/facturae/samples/invoice-complete.xml"
                "deps/corpus/es/facturae/samples/invoice-face.xml"
                "deps/corpus/es/facturae/samples/invoice-overseas-individual.xml"
                "deps/corpus/es/facturae/samples/invoice-usd.xml"
                "deps/corpus/es/facturae/samples/invoice-vat-irpf-inc.xml"
                "deps/corpus/es/facturae/samples/invoice-vat-irpf.xml"
                "deps/corpus/es/facturae/samples/invoice-vat-provider.xml"
                "deps/corpus/es/facturae/samples/invoice-vat.xml"
            ]
        ubl_json_documents = Listed []
        extras =
            [
                GateResolve
                GatePreload (repo_path "deps/corpus/eu/ubl21/common/UBL-xmldsig-core-schema-2.1.xsd")
            ]
    }

:= cii_d16b () : family_root =
    {
        module_name = module_id "generated_cii_d16b"
        schema_path =
            repo_path (
                "deps/corpus/eu/en16931-cii/cii/schema/D16B SCRDM (Subset)/uncoupled clm/CII/uncefact/data/standard/"
                ^^ "CrossIndustryInvoice_100pD16B.xsd"
            )
        root_element = tag_name "CrossIndustryInvoice"
        root_type = definition_name "CrossIndustryInvoiceType"
        target_namespace = namespace_uri "urn:un:unece:uncefact:data:standard:CrossIndustryInvoice:100"
        source = Xsd []
        known_defect = None
        documents =
            Listed [
                "deps/corpus/eu/en16931-cii/cii/examples/CII_business_example_01.xml"
                "deps/corpus/eu/en16931-cii/cii/examples/CII_business_example_02.xml"
                "deps/corpus/eu/en16931-cii/cii/examples/CII_business_example_Z.xml"
                "deps/corpus/eu/en16931-cii/cii/examples/CII_example1.xml"
                "deps/corpus/eu/en16931-cii/cii/examples/CII_example2.xml"
                "deps/corpus/eu/en16931-cii/cii/examples/CII_example3.xml"
                "deps/corpus/eu/en16931-cii/cii/examples/CII_example4.xml"
                "deps/corpus/eu/en16931-cii/cii/examples/CII_example5.xml"
                "deps/corpus/eu/en16931-cii/cii/examples/CII_example6.xml"
                "deps/corpus/eu/en16931-cii/cii/examples/CII_example7.xml"
                "deps/corpus/eu/en16931-cii/cii/examples/CII_example8.xml"
                "deps/corpus/eu/en16931-cii/cii/examples/CII_example9.xml"
                "deps/corpus/eu/en16931-cii/cii/examples/CII-BR-CO-10-RoundingIssue.xml"
                "deps/corpus/eu/en16931-cii/cii/examples/huf_example_cii.xml"
                "deps/corpus/eu/en16931-cii/cii/examples/XRechnung-O.xml"
            ]
        ubl_json_documents = Listed []
        extras = [ Ungated "its example has no schema gate yet; the CII family item adds one" ]
    }

:= verifactu_registro () : family_root =
    {
        module_name = module_id "generated_es_verifactu"
        schema_path = repo_path "deps/corpus/es/verifactu/SuministroLR.xsd"
        root_element = tag_name "RegFactuSistemaFacturacion"
        root_type = definition_name "RegFactuSistemaFacturacion"
        target_namespace =
            namespace_uri (
                "https://www2.agenciatributaria.gob.es/static_files/common/internet/dep/aplicaciones/es/aeat/tike/c"
                ^^ "ont/ws/SuministroLR.xsd"
            )
        source =
            Xsd [
                (
                    "http://www.w3.org/TR/xmldsig-core/xmldsig-core-schema.xsd",
                    repo_path "deps/corpus/eu/ubl21/common/UBL-xmldsig-core-schema-2.1.xsd"
                )
            ]
        known_defect = None
        documents = Listed []
        ubl_json_documents = Listed []
        extras =
            [
                GateResolve
                GatePreload (repo_path "deps/corpus/eu/ubl21/common/UBL-xmldsig-core-schema-2.1.xsd")
            ]
    }

:= saft_pt_1_04_01 () : family_root =
    {
        module_name = module_id "generated_pt_saft_1_04_01"
        schema_path = repo_path "deps/corpus/pt/saft/saftpt1.04_01.xsd"
        root_element = tag_name "AuditFile"
        root_type = definition_name "AuditFile"
        target_namespace = namespace_uri "urn:OECD:StandardAuditFile-Tax:PT_1.04_01"
        source = Xsd []
        known_defect = None
        documents = Listed [ "deps/corpus/pt/saft/samples/saft_idemo599999999.xml" ]
        ubl_json_documents = Listed []
        extras = [ Ungated "its example has no schema gate yet" ]
    }

:= mydata_2_0_2 () : family_root =
    {
        module_name = module_id "generated_gr_mydata_2_0_2"
        schema_path = repo_path "deps/corpus/gr/mydata/InvoicesDoc-v2.0.2.xsd"
        root_element = tag_name "InvoicesDoc"
        root_type = definition_name "InvoicesDoc"
        target_namespace = namespace_uri "http://www.aade.gr/myDATA/invoice/v1.0"
        source = Xsd []
        known_defect = None
        documents = Manifest (repo_path "deps/corpus/gr-mydata-invoicesdoc-documents.txt")
        ubl_json_documents = Listed []
        extras =
            [
                GateResolve
                GatePreload (repo_path "deps/corpus/eu/ubl21/common/UBL-xmldsig-core-schema-2.1.xsd")
            ]
    }

:= hacienda_factura_4_4 () : family_root =
    {
        module_name = module_id "generated_cr_factura_4_4"
        schema_path = repo_path "deps/corpus/cr/hacienda/v4.4/FacturaElectronica_V4.4.xsd"
        root_element = tag_name "FacturaElectronica"
        root_type = definition_name "FacturaElectronica"
        target_namespace =
            namespace_uri "https://cdn.comprobanteselectronicos.go.cr/xml-schemas/v4.4/facturaElectronica"
        source =
            Xsd [
                (
                    "../../xmldsig-core-schema.xsd",
                    repo_path "deps/corpus/eu/ubl21/common/UBL-xmldsig-core-schema-2.1.xsd"
                )
            ]
        known_defect = None
        documents = Listed []
        ubl_json_documents = Listed []
        extras =
            [
                Signed
                GateResolve
                GatePreload (repo_path "deps/corpus/eu/ubl21/common/UBL-xmldsig-core-schema-2.1.xsd")
            ]
    }

:= hacienda_namespace (root : string) : namespace_uri =
    := first = sm'::substring 0i32 1i32 root
    := rest = sm'::substring 1i32 (sm'::length root - 1i32) root
    namespace_uri ("https://cdn.comprobanteselectronicos.go.cr/xml-schemas/v4.4/" ++# sm'::to_lower first ++# rest)

:= xmldsig_import () : string * repo_path =
    "../../xmldsig-core-schema.xsd", repo_path "deps/corpus/eu/ubl21/common/UBL-xmldsig-core-schema-2.1.xsd"

:= hacienda_base (root : string) (suffix : string) : family_root =
    bare_family
        (module_id ("generated_cr_" ++# suffix ++# "_4_4"))
        (repo_path ("deps/corpus/cr/hacienda/v4.4/" ++# root ++# "_V4.4.xsd"))
        (tag_name root)
        (definition_name root)
        (hacienda_namespace root)

:= hacienda_document (root : string) (suffix : string) : family_root =
    := base = hacienda_base root suffix
    { base with source = Xsd [ xmldsig_import () ]; extras = Signed :: resolved_gate () }

:= hacienda_rows () : list (string * string) =
    [
        ("FacturaElectronicaCompra", "factura_compra"); ("FacturaElectronicaExportacion", "factura_exportacion");
        ("NotaCreditoElectronica", "nota_credito"); ("NotaDebitoElectronica", "nota_debito");
        ("TiqueteElectronico", "tiquete"); ("ReciboElectronicoPago", "recibo_pago");
        ("MensajeReceptor", "mensaje_receptor"); ("MensajeHacienda", "mensaje_hacienda")
    ]

:= hacienda_documents () : list family_root =
    listm::map (\ (root, suffix) => hacienda_document root suffix) (hacienda_rows ())

:= nav_invoice_data_3_0 () : family_root =
    {
        module_name = module_id "generated_hu_nav_3_0"
        schema_path = repo_path "deps/corpus/hu/nav/invoiceData.xsd"
        root_element = tag_name "InvoiceData"
        root_type = definition_name "InvoiceData"
        target_namespace = namespace_uri "http://schemas.nav.gov.hu/OSA/3.0/data"
        source =
            Xsd [
                ("http://schemas.nav.gov.hu/NTCA/1.0/common", repo_path "deps/corpus/hu/nav/common.xsd");
                ("http://schemas.nav.gov.hu/OSA/3.0/base", repo_path "deps/corpus/hu/nav/invoiceBase.xsd")
            ]
        known_defect = None
        documents = Manifest (repo_path "deps/corpus/hu-nav-invoicedata-documents.txt")
        ubl_json_documents = Listed []
        extras = [ Ungated "its example has no schema gate yet" ]
    }

:= gst_inv01_1_1 () : family_root =
    {
        module_name = module_id "generated_in_gst_inv01_1_1"
        schema_path = repo_path "deps/corpus/in/gst/inv01_1.1.json"
        root_element = tag_name "GstInvoice"
        root_type = definition_name "GstInvoice"
        target_namespace = namespace_uri ""
        source = JsonSchema
        known_defect = None
        documents = Listed [ "deps/corpus/in/gst/examples/irn-request-example.json" ]
        ubl_json_documents = Listed []
        extras = []
    }

:= single_families_americas () : list family_root =
    [
        nfe_v4_00 (); cfdi_v4_0 (); cfdi_tfd_1_1 (); cfdi_pagos_2_0 (); chile_dte_v10 (); chile_envio_dte_v10 ()
        hacienda_factura_4_4 ()
    ]

:= single_families_elsewhere () : list family_root =
    [
        ubl_invoice_2_1 (); ubl_creditnote_2_1 (); fatturapa_1_2_3 (); ksef_fa_3 (); facturae_3_2_2 ()
        verifactu_registro (); saft_pt_1_04_01 (); cii_d16b (); mydata_2_0_2 (); nav_invoice_data_3_0 ()
        gst_inv01_1_1 ()
    ]

:= all_families () : list family_root =
    listm::concat [
        family_roots (); nfe_events (); ubl_business_documents_2_1 (); hacienda_documents ()
        single_families_americas (); single_families_elsewhere ()
    ]

:= gate_example (family : family_root) : string =
    ?= family::source ?|
    | JsonSchema => module_text family::module_name ++# "_example.json"
    | Xsd _ => module_text family::module_name ++# "_example.xml"

:= gate_kind (family : family_root) : string =
    ?= family::source ?|
    | JsonSchema => "json"
    | Xsd _ => "xsd"

:= gate_flag (family : family_root) : string = ? has_extra GateResolve family::extras ?> "1" |: "0"

:= gate_row (family : family_root) : string =
    := preloads = join_with ";" (gate_preloads family::extras) ""
    := fields = [ gate_example family; repo_text family::schema_path; preloads; gate_flag family; gate_kind family ]
    join_with "\t" fields ""

:~ ^ gate_rows (families : list family_root) : list string =
    ?= families ?|
    | family :: rest => ? ungated_reason family::extras = None ?> gate_row family :: gate_rows rest |: gate_rows rest
    | [] => []

:= gate_table () : string =
    join_with "\n" ("example\tschema\tpreloads\tresolve\tkind" :: gate_rows (all_families ())) ""

:! brazil = ()
:! europe_ubl = ()
:! mexico = ()

:| regime_witness r =
    | Brazil :: regime_witness brazil
    | EuropeUbl :: regime_witness europe_ubl
    | Mexico :: regime_witness mexico

:! regime_family r = family_root

:= family_root_of \* r. ((regime_family root) : regime_family r) : family_root = root

:= brazilian_families () : list (regime_family brazil) =
    listm::map regime_family (listm::append (family_roots ()) (nfe_v4_00 () :: nfe_events ()))

:= ubl_families () : list (regime_family europe_ubl) =
    listm::map regime_family (ubl_invoice_2_1 () :: ubl_creditnote_2_1 () :: ubl_business_documents_2_1 ())

:= mexican_families () : list (regime_family mexico) = listm::map regime_family [ cfdi_v4_0 () ]

:= families_of \* r. (witness : regime_witness r) : list (regime_family r) =
    ?= witness ?|
    | Brazil => brazilian_families ()
    | EuropeUbl => ubl_families ()
    | Mexico => mexican_families ()

:= access_key_digits (_ : regime_family brazil) : i32 = 44i32
:~ ^ contains_name (n : string) (names : list string) : bool =
    ?= names ?|
    | m :: rest => m = n || contains_name n rest
    | [] => !-

:~ ^ has_duplicate (names : list string) : bool =
    ?= names ?|
    | n :: tail => contains_name n tail || has_duplicate tail
    | [] => !-

#### every_family_has_its_own_module_name (test)

In [1]:
///- --test
///> rust

:= names = listm::map (\ (f : family_root) => module_text f::module_name) (all_families ())
? has_duplicate names ?> failwith "two families would write the same generated unit"
? (listm::length names : i32) <> 45i32 ?> failwith "45 families: 42 gated examples plus CII, SAF-T and NAV"

0

#### the_gate_table_has_one_row_per_gated_family (test)

In [2]:
///- --test
///> rust

:= table = gate_table ()
? (listm::length (sm'::split_list "\n" table) : i32) <> 43i32 ?> failwith "a header and 42 gated families"
:= nfe = "generated_nfe_v4_example.xml\tdeps/corpus/DFe.NET/NFe.AppTeste/Schemas/nfe_v4.00.xsd\t\t0\txsd"
? sm'::contains nfe table = !- ?> failwith "a DFe family validates against its own schema alone"
:= dsig = "\tdeps/corpus/eu/ubl21/common/UBL-xmldsig-core-schema-2.1.xsd\t1\txsd"
? sm'::contains ("generated_cl_dte_v10_example.xml\tdeps/corpus/cl/dte/DTE_v10.xsd\t\t1\txsd") table = !-
?> failwith "Chile DTE resolves its own signature import"
? sm'::contains ("UBL-Invoice-2.1.xsd" ++# dsig) table = !- ?> failwith "UBL preloads the W3C xmldsig schema"
? sm'::contains "generated_in_gst_inv01_1_1_example.json\tdeps/corpus/in/gst/inv01_1.1.json\t\t0\tjson" table = !-
?> failwith "GST is a JSON Schema gate"
? sm'::contains "generated_cii_d16b" table ?> failwith "an Ungated family writes no row"

0

#### every_regime_lists_its_own_families (test)

In [3]:
///- --test
///> rust

? (listm::length (families_of Brazil) : i32) <> 16i32
?> failwith "12 NF-e/CT-e/MDF-e roots, NF-e 4.00 and its 3 events are Brazilian"
? (listm::length (families_of EuropeUbl) : i32) <> 6i32
?> failwith "Invoice, CreditNote and 4 business documents are UBL"
? (listm::length (families_of Mexico) : i32) <> 1i32 ?> failwith "CFDI 4.0 is Mexican"
? (listm::length (listm::map access_key_digits (families_of Brazil)) : i32) <> 16i32
?> failwith "every Brazilian family has a 44-digit access key"

0

#### a_ubl_family_takes_no_brazilian_access_key (test)

In [4]:
///- --test --fails "Unification"
///> rust

?= families_of EuropeUbl ?|
| f :: _ => access_key_digits f |> ignore
| [] => ()

failed to compile as expected ("Unification"): FatalError: File main has a type error somewhere in its path. /home/runner/work/fiscal/fiscal/src/families.spi:781:35 (main.spi:749:35): ETY-TermError: Unification failure. Got:      regime_family brazil Expected: regime_family europe_ubl In the definition: spiral_cells_test.
